# Fase 0: reconhecimento dos dados do VRA (ANAC)

Este notebook confere o formato dos arquivos mensais do VRA e responde às perguntas da Fase 0
(especificação inicial do projeto, um documento de planejamento não publicado):

1. Todos os meses seguem o mesmo layout do arquivo de agosto de 2026?
2. O pipeline consegue reproduzir os números de referência de agosto de 2026?
3. Algum mês tem `Código Justificativa` preenchido?
4. O que os dados dizem sobre `Código Tipo Linha`, `Código Autorização (DI)` e o fuso horário?
5. Qual é a defasagem de publicação, e os meses antigos são reatualizados?

Os arquivos ficam em `data/raw/vra/` (fora do git). As conclusões estão resumidas em `docs/dados.md`.
Este notebook é só exploração: a lógica de produção vai para `src/fdr/` na Fase 1.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)


def achar_raiz(inicio: Path) -> Path:
    """Sobe as pastas até achar o pyproject.toml, para o notebook rodar de qualquer lugar."""
    for pasta in [inicio, *inicio.parents]:
        if (pasta / "pyproject.toml").exists():
            return pasta
    raise FileNotFoundError("Não achei a raiz do projeto (pyproject.toml).")


RAIZ = achar_raiz(Path.cwd())
PASTA_VRA = RAIZ / "data" / "raw" / "vra"
ARQUIVOS = sorted(PASTA_VRA.glob("VRA_*.csv"))
print(f"{len(ARQUIVOS)} arquivos em {PASTA_VRA}:")
for arq in ARQUIVOS:
    print(f"  {arq.name}  ({arq.stat().st_size / 1e6:.1f} MB)")

6 arquivos em c:\Users\fabia\OneDrive\Área de Trabalho\flight-delay-radar\data\raw\vra:
  VRA_20221.csv  (10.2 MB)
  VRA_20236.csv  (11.1 MB)
  VRA_20265.csv  (11.7 MB)
  VRA_20266.csv  (11.5 MB)
  VRA_20267.csv  (12.7 MB)
  VRA_20268.csv  (12.5 MB)


## 1. Formato bruto de cada arquivo

Lemos os primeiros bytes de cada arquivo para conferir, sem depender de nenhuma biblioteca de CSV:
BOM, quebra de linha, a linha de metadado "Atualizado em" e o cabeçalho.

In [2]:
CABECALHO_ESPERADO = [
    "ICAO Empresa Aérea",
    "Número Voo",
    "Código Autorização (DI)",
    "Código Tipo Linha",
    "ICAO Aeródromo Origem",
    "ICAO Aeródromo Destino",
    "Partida Prevista",
    "Partida Real",
    "Chegada Prevista",
    "Chegada Real",
    "Situação Voo",
    "Código Justificativa",
]

linhas_formato = []
for arq in ARQUIVOS:
    bruto = arq.read_bytes()
    texto = bruto.decode("utf-8-sig")
    linha1, linha2 = texto.split("\r\n", 2)[:2]
    cabecalho = [c.strip('"') for c in linha2.split(";")]
    linhas_formato.append(
        {
            "arquivo": arq.name,
            "bom_utf8": bruto.startswith(b"\xef\xbb\xbf"),
            "crlf": b"\r\n" in bruto[:2000],
            "linha_1": linha1,
            "cabecalho_igual": cabecalho == CABECALHO_ESPERADO,
            "linhas_de_dados": texto.count("\r\n") - 2 + (not texto.endswith("\r\n")),
        }
    )
formato = pd.DataFrame(linhas_formato)
formato

,arquivo,bom_utf8,crlf,linha_1,cabecalho_igual,linhas_de_dados
0,VRA_20221.csv,True,True,Atualizado em: 2026-10-07,True,73465
1,VRA_20236.csv,True,True,Atualizado em: 2026-10-07,True,79762
2,VRA_20265.csv,True,True,Atualizado em: 2026-10-07,True,82120
3,VRA_20266.csv,True,True,Atualizado em: 2026-10-07,True,81119
4,VRA_20267.csv,True,True,Atualizado em: 2026-10-07,True,89499
5,VRA_20268.csv,True,True,Atualizado em: 2026-10-07,True,88064


## 2. Carga no DuckDB

Mesmas regras da especificação inicial (`docs/dados.md`, seção 2): pular a linha 1, separador `;`, tudo como
texto, `null` vira ausente e as datas são cortadas nos 19 primeiros caracteres (alguns meses trazem
sufixo de nanossegundos). Guardamos o nome do arquivo de origem para comparar os meses.

In [3]:
con = duckdb.connect()
padrao = (PASTA_VRA / "VRA_*.csv").as_posix()
con.execute(f"""
create or replace table bruto as
select
    regexp_extract(filename, 'VRA_(\\d+)\\.csv', 1) as arquivo,
    *
exclude (filename)
from read_csv('{padrao}', skip = 1, header = true, delim = ';',
              all_varchar = true, filename = true)
""")


def para_data(coluna: str) -> str:
    return f"try_strptime(nullif(left(\"{coluna}\", 19), 'null'), '%Y-%m-%d %H:%M:%S')"


con.execute(f"""
create or replace table voos as
select distinct
    arquivo,
    "ICAO Empresa Aérea"      as cia,
    "Número Voo"              as numero_voo,
    "Código Autorização (DI)" as di,
    "Código Tipo Linha"       as tipo_linha,
    "ICAO Aeródromo Origem"   as origem,
    "ICAO Aeródromo Destino"  as destino,
    "Situação Voo"            as situacao,
    "Código Justificativa"    as justificativa,
    "Partida Prevista"        as partida_prevista_txt,
    "Partida Real"            as partida_real_txt,
    "Chegada Prevista"        as chegada_prevista_txt,
    "Chegada Real"            as chegada_real_txt,
    {para_data("Partida Prevista")} as partida_prevista,
    {para_data("Partida Real")}     as partida_real,
    {para_data("Chegada Prevista")} as chegada_prevista,
    {para_data("Chegada Real")}     as chegada_real,
    strptime(arquivo, '%Y%m')       as inicio_mes
from bruto
""")
con.execute("""
create or replace view voos_atraso as
select *, date_diff('minute', partida_prevista, partida_real) as atraso_min
from voos
""")


def sql(consulta: str) -> pd.DataFrame:
    return con.sql(consulta).df()


sql("""
select b.arquivo, b.linhas, v.sem_duplicatas, b.linhas - v.sem_duplicatas as duplicatas
from (select arquivo, count(*) as linhas from bruto group by 1) b
join (select arquivo, count(*) as sem_duplicatas from voos group by 1) v using (arquivo)
order by arquivo
""")

,arquivo,linhas,sem_duplicatas,duplicatas
0,20221,73465,73465,0
1,20236,79762,79762,0
2,20265,82120,82102,18
3,20266,81119,81119,0
4,20267,89499,89488,11
5,20268,88064,88020,44


## 3. Situação, tipo de linha e código de autorização por mês

In [4]:
sql("pivot (select arquivo, situacao from voos) on situacao using count(*) order by arquivo")

,arquivo,CANCELADO,NÃO INFORMADO,REALIZADO
0,20221,3286,62,70117
1,20236,4712,53,74997
2,20265,2386,0,79716
3,20266,2892,0,78227
4,20267,2695,0,86793
5,20268,2547,0,85473


In [5]:
sql("pivot (select arquivo, tipo_linha from voos) on tipo_linha using count(*) order by arquivo")

,arquivo,C,G,I,N,X
0,20221,1654,1609,6330,63381,491
1,20236,2010,2137,9511,65455,649
2,20265,1137,1801,13285,65879,0
3,20266,1167,1846,13286,64820,0
4,20267,1299,1908,15982,70299,0
5,20268,1222,1939,14840,70019,0


In [6]:
sql("pivot (select arquivo, di from voos) on di using count(*) order by arquivo")

,arquivo,0,1,2,3,4,6,7,9,D,E
0,20221,69959,104,1701,309,345,422,226,290,76,33
1,20236,76437,352,1112,344,361,448,257,367,69,15
2,20265,79148,625,429,284,466,414,290,384,43,19
3,20266,78155,596,417,328,497,441,227,399,50,9
4,20267,85836,699,610,326,402,512,308,734,44,17
5,20268,84999,610,408,304,459,547,303,338,48,4


Quem opera cada tipo de linha ajuda a entender os códigos que não estão documentados
(`C`, `G` e `X`).

In [7]:
sql("""
select * from (
    select tipo_linha, cia, count(*) as voos
    from voos
    where tipo_linha in ('C', 'G', 'X')
    group by tipo_linha, cia
)
qualify row_number() over (partition by tipo_linha order by voos desc) <= 5
order by tipo_linha, voos desc
""")

,tipo_linha,cia,voos
0,C,GLO,3278
1,C,AZU,1968
2,C,SID,1355
3,C,TTL,1009
4,C,LTG,350
5,G,LTG,1615
6,G,QTR,1428
7,G,LCO,1306
8,G,GTI,999
9,G,TPA,896


In [8]:
# Situação "NÃO INFORMADO": em quais meses aparece e se tem horário real.
sql("""
select arquivo, tipo_linha, di, count(*) as voos,
       count(partida_real) as com_partida_real
from voos
where situacao = 'NÃO INFORMADO'
group by all
order by all
""")

,arquivo,tipo_linha,di,voos,com_partida_real
0,20221,I,0,36,0
1,20221,I,2,2,0
2,20221,N,0,24,0
3,20236,I,2,4,0
4,20236,N,0,49,0


## 4. Valores ausentes, nanossegundos e justificativas

In [9]:
sql("""
select
    arquivo,
    sum((partida_prevista_txt = 'null')::int) as partida_prevista_null,
    sum((partida_real_txt = 'null')::int) as partida_real_null,
    sum((partida_prevista_txt <> 'null' and partida_prevista is null)::int) as datas_invalidas,
    sum((greatest(length(partida_prevista_txt), length(partida_real_txt),
                  length(chegada_prevista_txt), length(chegada_real_txt)) > 19)::int)
        as linhas_com_nanossegundos,
    sum((situacao = 'REALIZADO' and partida_prevista is null and di = '0')::int)
        as realizado_sem_previsto_di0,
    sum((situacao = 'CANCELADO' and partida_real is not null)::int)
        as cancelado_com_partida_real
from voos
group by arquivo
order by arquivo
""")

,arquivo,partida_prevista_null,partida_real_null,datas_invalidas,linhas_com_nanossegundos,realizado_sem_previsto_di0,cancelado_com_partida_real
0,20221,2777.0,3348.0,0.0,0.0,0.0,0.0
1,20236,2637.0,4765.0,0.0,0.0,0.0,0.0
2,20265,2243.0,2386.0,0.0,6513.0,0.0,0.0
3,20266,2308.0,2892.0,0.0,6531.0,0.0,0.0
4,20267,2558.0,2695.0,0.0,6823.0,0.0,0.0
5,20268,2307.0,2547.0,0.0,6547.0,0.0,0.0


In [10]:
sql("select arquivo, justificativa, count(*) as voos from voos group by all order by all")

,arquivo,justificativa,voos
0,20221,N/A,73465
1,20236,N/A,79762
2,20265,N/A,82102
3,20266,N/A,81119
4,20267,N/A,89488
5,20268,N/A,88020


## 5. Bordas do mês

O mês de um voo é definido pela partida prevista. Aqui medimos quantos voos de cada arquivo têm
partida prevista fora do mês do arquivo, e quantas horas depois da virada.

In [11]:
sql("""
select
    arquivo,
    tipo_linha,
    count(*) as voos_fora_do_mes,
    max(date_diff('hour', inicio_mes + interval 1 month, partida_prevista)) as max_horas_depois,
    sum((partida_prevista >= inicio_mes + interval 1 month + interval 6 hour)::int)
        as alem_de_6h,
    sum((partida_prevista < inicio_mes)::int) as antes_do_mes
from voos
where partida_prevista >= inicio_mes + interval 1 month or partida_prevista < inicio_mes
group by all
order by all
""")

,arquivo,tipo_linha,voos_fora_do_mes,max_horas_depois,alem_de_6h,antes_do_mes
0,20221,C,2,1,0.0,0.0
1,20221,G,14,14,3.0,0.0
2,20221,I,4,26,4.0,0.0
3,20236,C,4,1,0.0,0.0
4,20236,G,22,36,10.0,0.0
5,20236,I,5,7,2.0,0.0
6,20236,N,2,0,0.0,0.0
7,20265,C,1,0,0.0,0.0
8,20265,G,11,19,5.0,0.0
9,20265,I,8,11,3.0,0.0


In [12]:
# O mesmo voo aparece em dois arquivos?
sql("""
select a.arquivo as arquivo_a, b.arquivo as arquivo_b, a.cia, a.numero_voo, a.tipo_linha,
       a.partida_prevista, a.partida_real as real_a, b.partida_real as real_b
from voos a
join voos b
  on a.cia = b.cia and a.numero_voo = b.numero_voo and a.origem = b.origem
 and a.partida_prevista = b.partida_prevista and a.arquivo < b.arquivo
""")

,arquivo_a,arquivo_b,cia,numero_voo,tipo_linha,partida_prevista,real_a,real_b
0,20267,20268,GLO,9610,C,2026-08-01 01:05:00,2026-08-01 02:21:00,2026-08-01 02:21:00


In [13]:
# A combinação companhia + número + partida prevista + origem + destino + DI é única?
sql("""
select arquivo, tipo_linha, count(*) as linhas_em_conflito
from voos
where (arquivo, cia, numero_voo, partida_prevista, origem, destino, di) in (
    select (arquivo, cia, numero_voo, partida_prevista, origem, destino, di)
    from voos
    where partida_prevista is not null
    group by all
    having count(*) > 1
)
group by all
order by all
""")

,arquivo,tipo_linha,linhas_em_conflito
0,20221,C,2
1,20265,C,4
2,20265,I,8
3,20266,C,16


## 6. Atrasos extremos (candidatos à quarentena)

Regra da seção 5.2: atraso fora de −120 a 1.440 minutos vai para a quarentena, e a quarentena
precisa ficar abaixo de 2% do mês.

In [14]:
sql("""
select
    arquivo,
    min(atraso_min) as menor_atraso,
    max(atraso_min) as maior_atraso,
    sum((atraso_min < -120)::int) as abaixo_de_menos_120,
    sum((atraso_min > 1440)::int) as acima_de_1440,
    round(100 * avg((atraso_min < -120 or atraso_min > 1440)::int), 3) as pct_quarentena
from voos_atraso
where situacao = 'REALIZADO' and atraso_min is not null
group by arquivo
order by arquivo
""")

,arquivo,menor_atraso,maior_atraso,abaixo_de_menos_120,acima_de_1440,pct_quarentena
0,20221,-2040,4739,175.0,42.0,0.322
1,20236,-1468,4309,97.0,18.0,0.159
2,20265,-4348,2865,33.0,8.0,0.053
3,20266,-1453,3894,29.0,11.0,0.053
4,20267,-1191,3706,19.0,27.0,0.055
5,20268,-1444,4548,25.0,33.0,0.070


## 7. Números de referência de agosto de 2026

Recorte: voos realizados, com partida prevista e real, depois de remover as duplicatas exatas
(números de referência da especificação inicial, versão 2.1). A célula falha se algum número não bater.

In [15]:
def taxas(filtro: str) -> pd.Series:
    return (
        con.sql(f"""
        select count(*) as voos,
               round(100 * avg((atraso_min > 15)::int), 1) as a15,
               round(100 * avg((atraso_min > 30)::int), 1) as a30,
               round(100 * avg((atraso_min > 60)::int), 1) as a60
        from voos_atraso
        where arquivo = '20268' and situacao = 'REALIZADO'
          and partida_prevista is not null and partida_real is not null
          and {filtro}
    """)
        .df()
        .iloc[0]
    )


REFERENCIA = {
    "Nacional (N)": ("tipo_linha = 'N'", 67782, 15.7, 7.5, 2.5),
    "Nacional regular (N, DI=0)": ("tipo_linha = 'N' and di = '0'", 67241, 15.7, 7.5, 2.5),
    "Internacional (I)": ("tipo_linha = 'I'", 13452, 26.0, 14.6, 6.2),
    "LATAM (TAM), nacional": ("tipo_linha = 'N' and cia = 'TAM'", 23588, 21.0, None, None),
    "Azul (AZU), nacional": ("tipo_linha = 'N' and cia = 'AZU'", 22969, 11.5, None, None),
    "Gol (GLO), nacional": ("tipo_linha = 'N' and cia = 'GLO'", 19984, 14.7, None, None),
}

linhas_ref = []
for nome, (filtro, voos_esp, a15, a30, a60) in REFERENCIA.items():
    obtido = taxas(filtro)
    esperado = {"voos": voos_esp, "a15": a15, "a30": a30, "a60": a60}
    ok = all(v is None or abs(obtido[k] - v) <= 0.1 for k, v in esperado.items())
    linhas_ref.append({"recorte": nome, **obtido.to_dict(), "bate": ok})

conferencia = pd.DataFrame(linhas_ref)
display(conferencia)

cancel_n = con.sql(
    "select round(100 * avg((situacao = 'CANCELADO')::int), 1) "
    "from voos where arquivo = '20268' and tipo_linha = 'N'"
).fetchone()[0]
print(f"Cancelamentos em voos nacionais, agosto de 2026: {cancel_n}%")

assert conferencia["bate"].all(), "Algum número de referência não bateu."
assert cancel_n == 1.6

,recorte,voos,a15,a30,a60,bate
0,Nacional (N),67782.0,15.7,7.5,2.5,True
1,"Nacional regular (N, DI=0)",67241.0,15.7,7.5,2.5,True
2,Internacional (I),13452.0,26.0,14.6,6.2,True
3,"LATAM (TAM), nacional",23588.0,21.0,9.6,2.5,True
4,"Azul (AZU), nacional",22969.0,11.5,6.6,2.9,True
5,"Gol (GLO), nacional",19984.0,14.7,6.3,2.1,True


Cancelamentos em voos nacionais, agosto de 2026: 1.6%


## 8. Os mesmos indicadores em todos os meses

In [16]:
sql("""
select
    arquivo,
    count(*) filter (where tipo_linha = 'N' and di = '0') as voos_base_modelo,
    round(100 * avg((atraso_min > 15)::int) filter (where tipo_linha = 'N' and di = '0'), 1)
        as a15_base_modelo,
    round(100 * avg((atraso_min > 15)::int) filter (where tipo_linha = 'I'), 1)
        as a15_internacional
from voos_atraso
where situacao = 'REALIZADO' and partida_prevista is not null and partida_real is not null
group by arquivo
order by arquivo
""").merge(
    sql("""
        select arquivo,
               round(100 * avg((situacao = 'CANCELADO')::int), 2) as pct_cancelados_nacional
        from voos where tipo_linha = 'N' group by arquivo
    """),
    on="arquivo",
)

,arquivo,voos_base_modelo,a15_base_modelo,a15_internacional,pct_cancelados_nacional
0,20221,58501,15.4,28.9,3.71
1,20236,61124,14.5,28.7,3.69
2,20265,63393,13.3,22.3,1.41
3,20266,61913,17.4,23.5,1.97
4,20267,67005,14.5,26.0,1.67
5,20268,67241,15.7,26.0,1.61


## 9. Fuso horário

Se os horários estivessem no horário local de cada aeroporto, a duração prevista de Guarulhos para
Manaus seria cerca de 2 horas menor que a de Manaus para Guarulhos (Manaus fica 1 hora atrás de
Brasília). Se forem parecidas, os dois lados estão no mesmo fuso.

In [17]:
sql("""
select origem || ' → ' || destino as rota,
       count(*) as voos,
       median(date_diff('minute', partida_prevista, chegada_prevista)) as duracao_prevista_mediana
from voos
where (origem, destino) in (('SBGR', 'SBEG'), ('SBEG', 'SBGR'), ('SBGR', 'SCEL'), ('SCEL', 'SBGR'))
  and partida_prevista is not null and chegada_prevista is not null
group by all
order by rota
""")

,rota,voos,duracao_prevista_mediana
0,SBEG → SBGR,1221,235.0
1,SBGR → SBEG,1164,235.0
2,SBGR → SCEL,2413,260.0
3,SCEL → SBGR,2109,235.0


In [18]:
# Partidas nacionais por hora: o vale da madrugada indica qual é o fuso.
por_hora = sql("""
pivot (
    select arquivo, hour(partida_prevista) as hora
    from voos where tipo_linha = 'N' and partida_prevista is not null
) on arquivo using count(*) order by hora
""")
por_hora.head(8)

,hora,20221,20236,20265,20266,20267,20268
0,0,424,398,292,314,399,301
1,1,257,191,141,200,291,158
2,2,822,802,919,1001,1271,1189
3,3,1053,1117,1457,1490,1608,1491
4,4,1115,809,969,1004,1121,1022
5,5,2163,2258,2083,2154,2495,2441
6,6,3438,3176,3065,2878,3434,3127
7,7,2028,2294,2692,2564,2469,2755


## 10. Data de atualização e defasagem de publicação

In [19]:
formato[["arquivo", "linha_1"]]

,arquivo,linha_1
0,VRA_20221.csv,Atualizado em: 2026-10-07
1,VRA_20236.csv,Atualizado em: 2026-10-07
2,VRA_20265.csv,Atualizado em: 2026-10-07
3,VRA_20266.csv,Atualizado em: 2026-10-07
4,VRA_20267.csv,Atualizado em: 2026-10-07
5,VRA_20268.csv,Atualizado em: 2026-10-07


## Conclusões

O resumo das descobertas, das decisões e das diferenças em relação à especificação inicial está em
`docs/dados.md`.